# Hard-wall entropy and charge: lane A

This lane runs the cycle-resolved $N_y=40$ case and the endpoint-only $N_y=60$ case. Lane B owns every other size, so both notebooks may run concurrently.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Configuration

The scientific contract is locked. Runtime controls may be changed without changing the ensemble.

In [ ]:
from pathlib import Path

LANE = 'A'
BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/05_hard_wall_entropy_charge')
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs/hard_wall_entropy_charge_nx20_ny30-60_s100_2ny_raster_v1')
LOCAL_BUNDLE_DIR = Path('/content/05_hard_wall_entropy_charge')
SCRATCH_ROOT = Path('/content/hard_wall_entropy_charge_lane_A_scratch')
REPORT_ONLY = False
MAX_NEW_EXECUTION_BATCHES = None
RUN_ANALYSIS = False  # Enable only after the combined report is 140/140.

CONFIG = {
    'sampling_revision': 'hard_wall_entropy_charge_nx20_ny30-60_s100_2ny_raster_v1',
    'root_seed': 2026090305,
    'Nx': 20,
    'Ny_values': [30, 35, 40, 45, 50, 55, 60],
    'lane_Ny_values': {'A': [40, 60], 'B': [30, 35, 45, 50, 55]},
    'samples_per_Ny': 100,
    'result_shard_size': 5,
    'execution_batch_size_by_Ny': {'30': 80, '35': 60, '40': 40, '45': 30, '50': 25, '55': 20, '60': 20},
    'cycles_rule': '2*Ny',
    'segment_cycles': 5,
    'device': 'cuda:0',
    'dtype': 'complex128',
    'protocol': {
        'DW': True,
        'domain_wall_interval': [5, 15],
        'dw_truncation': True,
        'meas_slab_only': True,
        'nshell': 1,
        'alpha_1': 1.0,
        'alpha_2': 30.0,
        'filling_frac': 0.5,
        'trial_orbitals': 'X',
        'init_mode': 'default',
        'sequence': 'raster_y',
        'perfect_correction': True,
        'postselect': False,
        'postselect_probability': 0.0,
        'n_a': 0.5,
        'state_representation': 'physical_frame',
        'triv_region_local_mode': False,
    },
    'observer': {
        'detailed_Ny': 40,
        'fit_start_cycle': 10,
        'fit_min_Ay': 8,
        'wall_window_half_width': 1,
        'bootstrap_replicates': 20000,
        'bootstrap_seed': 2026090306,
    },
}
print(f'Lane: {LANE}; Ny={CONFIG["lane_Ny_values"][LANE]}')
print(f'Bundle: {BUNDLE_DRIVE_DIR}')
print(f'Output: {OUTPUT_ROOT}')
print('Lane A workload: 8 execution batches, 40 durable five-trajectory shards, 200 trajectories')

## 2. A100 and precision check

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required')
device_name = torch.cuda.get_device_name(0)
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
if 'A100' not in device_name or total_gib < 39.0:
    raise RuntimeError(f'An A100 40-GB-class GPU is required; found {device_name}, {total_gib:.1f} GiB')
if CONFIG['dtype'] != 'complex128':
    raise RuntimeError('Production dtype must remain complex128')
print(f'Device: {device_name}; memory={total_gib:.1f} GiB; dtype={CONFIG["dtype"]}')

## 3. Stage locally and run or resume

The child output stream is forwarded byte-for-byte so both durable-shard and cycle progress bars remain visible.

In [ ]:
import codecs
import json
import os
import shutil
import subprocess
import sys

def run_streaming_child(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    assert process.stdout is not None
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    try:
        while True:
            raw = os.read(process.stdout.fileno(), 4096)
            if not raw:
                break
            sys.stdout.write(decoder.decode(raw))
            sys.stdout.flush()
        tail = decoder.decode(b'', final=True)
        if tail:
            sys.stdout.write(tail)
            sys.stdout.flush()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

if not BUNDLE_DRIVE_DIR.is_dir():
    raise FileNotFoundError(f'Missing Drive bundle: {BUNDLE_DRIVE_DIR}')
if LOCAL_BUNDLE_DIR.exists():
    shutil.rmtree(LOCAL_BUNDLE_DIR)
shutil.copytree(BUNDLE_DRIVE_DIR, LOCAL_BUNDLE_DIR)
config_path = Path(f'/content/hard_wall_entropy_charge_lane_{LANE}_config.json')
config_path.write_text(json.dumps(CONFIG, indent=2, sort_keys=True) + '\n', encoding='utf-8')
command = [sys.executable, '-u', str(LOCAL_BUNDLE_DIR / 'run_campaign.py'), '--config', str(config_path), '--lane', LANE, '--output-root', str(OUTPUT_ROOT), '--scratch-root', str(SCRATCH_ROOT)]
if REPORT_ONLY:
    command.append('--report-only')
if MAX_NEW_EXECUTION_BATCHES is not None:
    command.extend(['--max-new-execution-batches', str(int(MAX_NEW_EXECUTION_BATCHES))])
print('[launch]', ' '.join(command), flush=True)
run_streaming_child(command)
print('[notebook] lane A runner exited successfully', flush=True)

## 4. Optional combined analysis

Enable only after a report-only pass confirms all 140 durable shards across both lanes.

In [ ]:
if RUN_ANALYSIS:
    analysis_root = OUTPUT_ROOT / 'analysis_outputs'
    run_streaming_child([sys.executable, '-u', str(LOCAL_BUNDLE_DIR / 'analyze_campaign.py'), '--output-root', str(OUTPUT_ROOT), '--analysis-root', str(analysis_root)])
else:
    print('Analysis skipped (RUN_ANALYSIS=False)')

## 5. Release the runtime

In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')